# CityDreamer → Berruguete (Tetuán, Madrid)

Genera un vídeo aéreo **generativo** del barrio a partir de su plano real de OpenStreetMap.

- GPU: **T4 (16 GB)** o superior. En Kaggle: *Accelerator → GPU T4 x1*, y activa Internet en las opciones del cuaderno.
- Pesos: HuggingFace (`hzxie/city-dreamer`), gratis y sin captcha.
- Plano: lo descarga de `Ntizar/berruguete-citydreamer` (hf.png + seg.png, 4171×3773 px, 1 px = 1 m).


In [ ]:
!nvidia-smi
!python -V
!free -g | head -2

## 1. Código y pesos

In [ ]:
import os
if not os.path.exists('CityDreamer'):
    !git clone -q https://github.com/hzxie/CityDreamer.git
os.chdir('CityDreamer')
!git apply /kaggle/working/inference-osm.patch 2>/dev/null || echo "parche no aplicado (¿ya estaba?)"
!mkdir -p output
%cd output
!curl -sL -o CityDreamer-Bgnd.pth "https://huggingface.co/hzxie/city-dreamer/resolve/main/CityDreamer-Bgnd.pth?download=true"
!curl -sL -o CityDreamer-Fgnd.pth "https://huggingface.co/hzxie/city-dreamer/resolve/main/CityDreamer-Fgnd.pth?download=true"
%cd ..
!ls -la output/

## 2. Dependencias y extensiones CUDA

Las tres extensiones son C++/CUDA: si esto falla, mira la versión de torch que trae la imagen.

In [ ]:
!pip install -q easydict einops flask flask-executor lxml matplotlib mayavi numpy opencv-python pillow pynvml scipy tqdm xvfbwrapper wandb "git+https://github.com/pvigier/perlin-numpy"
!nvidia-smi --query-gpu=name,memory.total --format=csv
!nvcc --version | tail -2
import torch; print('torch', torch.__version__, 'cuda', torch.version.cuda)

In [ ]:
import subprocess, glob, os
for d in sorted(glob.glob('extensions/*/')):
    print('===', d, flush=True)
    r = subprocess.run(['pip','install','-q','.'], cwd=d, capture_output=True, text=True)
    print(r.stdout[-1500:], r.stderr[-1500:])
import importlib
for m in ('extensions.extrude_tensor','extensions.voxlib','extensions.grid_encoder'):
    try:
        importlib.import_module(m); print('OK', m)
    except Exception as e:
        print('FALLO', m, e)

## 3. El plano de Berruguete

In [ ]:
!rm -rf /tmp/b && git clone -q --depth 1 https://github.com/Ntizar/berruguete-citydreamer /tmp/b
!mkdir -p data/osm && cp -r /tmp/b/layout data/osm/ES-Madrid-Berruguete && rm -f data/osm/ES-Madrid-Berruguete/*.png.orig
!ls -la data/osm/ES-Madrid-Berruguete
import numpy as np
from PIL import Image
hf = np.array(Image.open('data/osm/ES-Madrid-Berruguete/hf.png'))
seg = np.array(Image.open('data/osm/ES-Madrid-Berruguete/seg.png').convert('P'))
print('hf', hf.shape, hf.dtype, 'max', hf.max())
print('seg', seg.shape, 'clases', np.unique(seg, return_counts=True)[0])
assert hf.shape[0] >= 2880 and seg.shape[0] >= 2880, 'el layout debe ser >= 2880 px'

## 4. Render

`--patch_height 3` es conservador para 16 GB. Sube a 4-5 si no revienta la memoria. `--n_frames 8` para la primera prueba; 24 es el vídeo completo (tarda bastante más: el coste es por edificio visible y por frame).

In [ ]:
!python3 scripts/inference.py   --city_osm_dir data/osm/ES-Madrid-Berruguete   --patch_height 3 --patch_width 3   --n_frames 8   --output_file output/berruguete.mp4 2>&1 | tail -40
!ls -la output/*.mp4

## 5. Ver el resultado

In [ ]:
import os
from IPython.display import Video
p = 'output/berruguete.mp4'
print(os.path.getsize(p)/1e6, 'MB')
Video(p, embed=True, width=960)

## Notas

- Si sale **OOM**: baja `--patch_height/--patch_width` a 2 y `--n_frames` a 4.
- Si el vídeo sale **plano**: es lo esperado con un barrio de 6 plantas. El modelo está entrenado
  con rascacielos y normaliza la altura a 640 m.
- El `.mp4` queda en `/kaggle/working/CityDreamer/output/` para descargar.
